# Perceptrón multicapa para regresión — `MLPRegressor` sobre California Housing

**Módulo 06 — Fundamentos de redes neuronales.** Recurso descargable de la clase (`OD_RN1_ESP_M03_S11`), ampliado con explicaciones, análisis de los datos y lectura de las métricas.

## Objetivo

Predecir el **valor mediano de una vivienda** en un distrito de California a partir de 8 características del distrito, usando un perceptrón multicapa.

## Qué cambia respecto de la clasificación

Este es el mismo modelo del notebook anterior, pero resolviendo el **otro** tipo de problema. Las diferencias son concretas:

| | `MLPClassifier` | `MLPRegressor` |
|---|---|---|
| Qué predice | una **clase** (etiqueta discreta) | un **valor continuo** |
| Capa de salida | una neurona por clase | **una sola** neurona, sin activación |
| Función de pérdida | log-loss (entropía cruzada) | **error cuadrático medio** |
| Métricas | exactitud, precision, recall, F1 | **MSE**, **RMSE**, **R²** |

La arquitectura interna (capas ocultas, activación no lineal, backpropagation) es idéntica. Ver [`Perceptrón Multicapa.md`](../teoria/Perceptrón%20Multicapa.md).

## Qué se agregó al recurso original

El notebook de la clase eran 12 celdas de código sin texto. Acá se suma: descripción del dataset y sus unidades, la explicación de **por qué el escalado es obligatorio** en este caso (y no opcional como en Iris), RMSE en unidades interpretables, la lectura de qué significa R², el análisis del **techo artificial** del dataset —que explica la banda rara del gráfico de dispersión— y una comparación contra un modelo lineal de referencia.

## 1. Librerías

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42

## 2. El dataset California Housing

20.640 distritos censales de California (censo de 1990). Cada fila es un **distrito**, no una casa.

| Columna | Significado |
|---|---|
| `MedInc` | ingreso mediano del distrito (en decenas de miles de dólares) |
| `HouseAge` | antigüedad mediana de las viviendas (años) |
| `AveRooms` | promedio de habitaciones por vivienda |
| `AveBedrms` | promedio de dormitorios por vivienda |
| `Population` | población del distrito |
| `AveOccup` | promedio de ocupantes por vivienda |
| `Latitude` / `Longitude` | ubicación geográfica |
| **`MedHouseVal`** | **objetivo:** valor mediano de la vivienda, en cientos de miles de dólares |

La unidad del objetivo importa para leer las métricas: `MedHouseVal = 2.5` significa **250.000 dólares**.

> La primera ejecución de `fetch_california_housing()` **descarga** el dataset y lo cachea en `~/scikit_learn_data`. Requiere conexión.

In [2]:
california = fetch_california_housing()

data_df = pd.DataFrame(california.data, columns=california.feature_names)
data_df["MedHouseVal"] = california.target

print("Forma:", data_df.shape)
data_df.head()

Forma: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 3. Inspección: tipos y datos faltantes

Paso obligatorio antes de modelar. Un solo `NaN` hace fallar el `fit`.

In [3]:
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [4]:
print("Datos faltantes por columna:")
print(data_df.isna().sum().sort_values(ascending=False))

print("\nEstadisticas del objetivo (MedHouseVal):")
print(data_df["MedHouseVal"].describe())

Datos faltantes por columna:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Estadisticas del objetivo (MedHouseVal):
count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64


**Detalle clave que conviene notar ahora:** el máximo de `MedHouseVal` es exactamente **5.00001**. No es casualidad — el dataset original **recortó** todos los valores por encima de 500.000 dólares a ese tope. Son **992 distritos** (casi el 5% del dataset) con ese valor idéntico.

Esto va a aparecer más adelante como una **banda horizontal** en el gráfico de predicciones, y es un límite del **dato**, no del modelo: ninguna red puede predecir bien un valor que fue artificialmente truncado.

## 4. Explorar la relación de cada atributo con el objetivo

In [5]:
plt.figure(figsize=(18, 9))

for i, col in enumerate(["MedInc", "HouseAge", "AveRooms", "Population", "AveOccup", "Latitude"], start=1):
    plt.subplot(2, 3, i)
    sns.scatterplot(x=col, y="MedHouseVal", data=data_df, s=6, alpha=0.3)
    plt.title(f"{col} vs MedHouseVal")

plt.tight_layout()
plt.show()

/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50212/2530456780.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Tres cosas para leer en estos gráficos:

1. **`MedInc` es el atributo más informativo:** la nube sube claramente. Donde la gente gana más, las casas valen más — razonable.
2. **La banda horizontal en 5.0** aparece en todos los paneles: es el techo artificial de la sección 3.
3. **Las relaciones no son rectas** y hay atributos con valores extremos enormes (`AveRooms`, `AveOccup`, `Population`). Esa no linealidad es justamente lo que justifica usar un MLP en lugar de una regresión lineal.

## 5. Correlaciones

In [6]:
plt.figure(figsize=(9, 7))
sns.heatmap(data_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlacion")
plt.show()

print("Correlacion de cada atributo con el objetivo:")
print(data_df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False))

Correlacion de cada atributo con el objetivo:
MedInc        0.688075
AveRooms      0.151948
HouseAge      0.105623
AveOccup     -0.023737
Population   -0.024650
Longitude    -0.045967
AveBedrms    -0.046701
Latitude     -0.144160
Name: MedHouseVal, dtype: float64


/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50212/2331278764.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`MedInc` domina con amplitud. Notar también la fuerte correlación negativa entre `Latitude` y `Longitude` (**-0.93**): no es un hallazgo del negocio sino pura geografía, California es un estado alargado en diagonal.

## 6. Atributos y objetivo

In [7]:
X = data_df.drop(columns="MedHouseVal").values
y = data_df["MedHouseVal"].values

print("X:", X.shape, "| y:", y.shape)
print("Atributos:", list(california.feature_names))

X: (20640, 8) | y: (20640,)
Atributos: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## 7. División train / test

Sin `stratify`: no aplica en regresión, porque el objetivo es continuo y no hay clases que balancear.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA
)

print("Entrenamiento:", X_train.shape[0], "| Prueba:", X_test.shape[0])

Entrenamiento: 16512 | Prueba: 4128


## 8. Escalado: acá no es opcional

En Iris los dos atributos estaban en centímetros y en rangos parecidos, así que el escalado era buena práctica pero no cambiaba mucho. **Acá es imprescindible**, y se ve en los números: `Population` llega a decenas de miles mientras que `AveBedrms` ronda 1. Sin escalar, el gradiente queda dominado por los atributos de magnitud grande y el entrenamiento no converge de forma razonable.

Otra vez: `fit_transform` en train, solo `transform` en test.

In [9]:
print("Rangos ANTES de escalar (min / max por atributo):")
print(pd.DataFrame({"min": X_train.min(axis=0).round(2),
                    "max": X_train.max(axis=0).round(2)},
                   index=california.feature_names))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nDespues de escalar, media ~0 y desvio ~1:")
print(pd.DataFrame({"media": X_train_scaled.mean(axis=0).round(3),
                    "desvio": X_train_scaled.std(axis=0).round(3)},
                   index=california.feature_names))

Rangos ANTES de escalar (min / max por atributo):
               min       max
MedInc        0.50     15.00
HouseAge      1.00     52.00
AveRooms      0.89    141.91
AveBedrms     0.33     25.64
Population    3.00  35682.00
AveOccup      0.69   1243.33
Latitude     32.55     41.95
Longitude  -124.35   -114.31

Despues de escalar, media ~0 y desvio ~1:
            media  desvio
MedInc       -0.0     1.0
HouseAge     -0.0     1.0
AveRooms      0.0     1.0
AveBedrms    -0.0     1.0
Population   -0.0     1.0
AveOccup     -0.0     1.0
Latitude      0.0     1.0
Longitude    -0.0     1.0


## 9. Entrenar el MLPRegressor

| Parámetro | Qué hace |
|---|---|
| `hidden_layer_sizes=(100, 100)` | dos capas ocultas de 100 neuronas |
| `max_iter=1000` | tope de épocas |
| `alpha=0.001` | **regularización L2**: penaliza pesos grandes para contener el sobreajuste |
| `random_state` | reproducibilidad |

`alpha` es el parámetro que responde a la advertencia de la teoría sobre el riesgo de sobreajuste: cuanto más alto, más "suave" y menos flexible el modelo.

> Esta celda tarda: son 16.512 muestras por hasta 1000 épocas.

In [10]:
mlp = MLPRegressor(
    hidden_layer_sizes=(100, 100),
    max_iter=1000,
    alpha=0.001,
    random_state=SEMILLA,
)
mlp.fit(X_train_scaled, y_train)

print("Epocas ejecutadas:", mlp.n_iter_, "de", mlp.max_iter)
print("Perdida final (MSE interno):", round(mlp.loss_, 4))

Epocas ejecutadas: 256 de 1000
Perdida final (MSE interno): 0.0906


In [11]:
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_)
plt.title("Curva de perdida durante el entrenamiento")
plt.xlabel("Epoca")
plt.ylabel("Perdida (MSE)")
plt.grid()
plt.show()

/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50212/3152343257.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Predecir y evaluar

Las tres métricas de regresión que usamos, y cómo se leen:

- **MSE** (error cuadrático medio): promedio de `(real − predicho)²`. Está en unidades **al cuadrado**, así que no es directamente interpretable; sirve para comparar modelos entre sí. Penaliza fuerte los errores grandes.
- **RMSE**: la raíz del MSE. Vuelve a las **unidades del objetivo**, así que sí se puede leer: "el modelo se equivoca en promedio en tanto".
- **R²**: qué proporción de la varianza del objetivo explica el modelo. `1.0` es perfecto, `0.0` equivale a predecir siempre la media, y **negativo** significa que el modelo es peor que predecir la media.

Agregamos también el **MAE** (error absoluto medio), que es más robusto a los valores extremos que el RMSE.

In [12]:
y_pred = mlp.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}   -> unos {rmse * 100_000:,.0f} dolares de error tipico")
print(f"MAE:  {mae:.4f}   -> unos {mae * 100_000:,.0f} dolares de error absoluto medio")
print(f"R2:   {r2:.4f}   -> el modelo explica el {r2 * 100:.1f}% de la varianza")

MSE:  0.2632
RMSE: 0.5131   -> unos 51,306 dolares de error tipico
MAE:  0.3505   -> unos 35,051 dolares de error absoluto medio
R2:   0.7991   -> el modelo explica el 79.9% de la varianza


## 11. Comparar contra una referencia

Un número aislado no dice si el modelo es bueno. Hacen falta dos referencias:

- **El modelo trivial:** predecir siempre la media. Por definición tiene R² = 0.
- **Una regresión lineal:** el modelo simple e interpretable. Si el MLP no le gana, toda su complejidad no se justifica.

In [13]:
lineal = LinearRegression().fit(X_train_scaled, y_train)
pred_lineal = lineal.predict(X_test_scaled)

media = np.full_like(y_test, y_train.mean())

comparacion = pd.DataFrame([
    {"modelo": "Predecir la media", "RMSE": np.sqrt(mean_squared_error(y_test, media)),
     "MAE": mean_absolute_error(y_test, media), "R2": r2_score(y_test, media)},
    {"modelo": "Regresion lineal", "RMSE": np.sqrt(mean_squared_error(y_test, pred_lineal)),
     "MAE": mean_absolute_error(y_test, pred_lineal), "R2": r2_score(y_test, pred_lineal)},
    {"modelo": "MLPRegressor (100, 100)", "RMSE": rmse, "MAE": mae, "R2": r2},
])
comparacion.round(4)

,modelo,RMSE,MAE,R2
0,Predecir la media,1.1449,0.9061,-0.0002
1,Regresion lineal,0.7456,0.5332,0.5758
2,"MLPRegressor (100, 100)",0.5131,0.3505,0.7991


Esta tabla es la que justifica (o no) haber usado una red neuronal. Lo esperable es que el MLP le gane a la regresión lineal, porque las relaciones que vimos en la sección 4 **no son lineales** — y eso es precisamente lo que el modelo lineal no puede capturar y el MLP sí.

## 12. Reales vs predicciones

El gráfico de diagnóstico estándar en regresión. Cada punto es un distrito del conjunto de prueba. La diagonal roja es la predicción perfecta: cuanto más pegados a ella estén los puntos, mejor.

In [14]:
plt.figure(figsize=(9, 8))
sns.scatterplot(x=y_test, y=y_pred, s=8, alpha=0.25)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         "--r", linewidth=2, label="prediccion perfecta")

plt.xlabel("Valores reales (MedHouseVal)")
plt.ylabel("Predicciones")
plt.title("Valores reales vs predicciones")
plt.legend()
plt.grid()
plt.show()

/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50212/3983187563.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Cómo leerlo:**

- La nube sigue la diagonal: el modelo capta la tendencia general.
- **La columna vertical de puntos en x = 5.0** es el techo artificial de la sección 3. Son los distritos truncados a 500.000 dólares: el modelo predice valores repartidos por debajo porque no tiene forma de saber cuánto valían realmente. Es un límite del dato, no una falla del modelo.
- La dispersión crece en los valores altos: el modelo es más confiable en las viviendas de valor medio, donde tiene más ejemplos.

## 13. Distribución de los residuos

El **residuo** es `real − predicho`. Su distribución dice si el modelo está sesgado.

In [15]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(residuos, bins=60, kde=True, ax=axes[0])
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Distribucion de los residuos")
axes[0].set_xlabel("Residuo (real - predicho)")

axes[1].scatter(y_pred, residuos, s=6, alpha=0.25)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residuos vs predicciones")
axes[1].set_xlabel("Prediccion")
axes[1].set_ylabel("Residuo")

plt.tight_layout()
plt.show()

print(f"Media de los residuos: {residuos.mean():.4f}  (idealmente cerca de 0)")
print(f"Desvio de los residuos: {residuos.std():.4f}")

Media de los residuos: -0.0539  (idealmente cerca de 0)
Desvio de los residuos: 0.5102


/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50212/1480419505.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Qué buscar:** residuos centrados en 0 y repartidos parejo. Si estuvieran corridos, el modelo tendría un **sesgo** sistemático (subestima o sobreestima). Si en el panel derecho se viera un patrón —un embudo, una curva—, sería señal de que el modelo no capturó parte de la estructura de los datos.

Acá la media de los residuos da levemente **negativa** (alrededor de -0.05, unos 5.000 dólares sobre un objetivo que promedia 207.000): el modelo **sobreestima apenas**, un sesgo chico pero sistemático. La cola izquierda del histograma es más larga que la derecha, y son justamente los distritos del techo en 5.0, donde el modelo predice por debajo del valor recortado.

## 14. Conclusiones

- `MLPRegressor` es el mismo modelo que `MLPClassifier` con otra capa de salida y otra función de pérdida; todo lo demás (capas ocultas, activación no lineal, backpropagation) es idéntico.
- En regresión, **`StandardScaler` no es opcional** cuando los atributos tienen escalas dispares, como acá.
- El **RMSE** es la métrica interpretable (está en las unidades del objetivo); el **R²** dice cuánta varianza se explica; el **MSE** sirve para comparar.
- Una métrica sola no alcanza: hay que compararla contra un **baseline** (la media, un modelo lineal) para saber si el modelo aporta algo.
- **Conocer el dataset importa tanto como el modelo.** El techo artificial en 500.000 dólares limita el rendimiento alcanzable y no se descubre mirando métricas: se descubre mirando los datos.
- Los gráficos de reales vs predicciones y de residuos muestran **dónde** falla el modelo, cosa que un número agregado nunca dice.

## 15. Para seguir practicando

1. Filtrar los distritos con `MedHouseVal >= 5.0` (el techo artificial) y reentrenar. ¿Mejoran las métricas?
2. Entrenar **sin escalar** (`X_train` en lugar de `X_train_scaled`) y comparar `n_iter_`, el RMSE y la curva de pérdida. Es la demostración de por qué el escalado importa.
3. Subir `alpha` a `1.0` y a `10.0`. ¿Cómo cambia la diferencia entre el error de entrenamiento y el de prueba?
4. Probar `hidden_layer_sizes=(50,)` y `(200, 200, 200)`. ¿Compensa el costo extra?
5. Agregar `early_stopping=True` y comparar `n_iter_` contra el modelo actual.